In [33]:
import os
import glob
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter
import mplhep as hep
from coffea.util import load

# ==========================================
# 1. GLOBAL CONFIGURATION & VARIABLES
# ==========================================
COFFEA_DIR = "DataVsMCTrigEff/" # <-- UPDATE THIS PATH

all_processes = ['TTBar', 'data_obs', 'tt_B'] 

weight_vars = [
    'events_genWeight', 'events_norm_weight', 'events_topptWeight', 
    'events_ele_reco_sf', 'events_ele_id_sf', 'events_mu_id_sf', 'events_mu_iso_sf', 
    'events_puWeight', 'events_mu_trig_sf'
]

mu_triggers = [
    'HLT_IsoMu24', 
    'HLT_Mu50', 
    'HLT_HighPtTkMu100', 
    'HLT_CascadeMu100'
]

ele_triggers = [
    'HLT_Ele30_WPTight_Gsf', 
    'HLT_Ele115_CaloIdVT_GsfTrkIdT', 
    'HLT_Ele50_CaloIdVT_GsfTrkIdT_PFJet165'
]

trigger_vars = mu_triggers + ele_triggers

validation_vars = [
    'events_ele_pt', 'events_ele_eta', 'events_n_ak4jets', 'events_n_b_outZH'
] + trigger_vars

vars_to_extract = validation_vars + weight_vars

# --- BINNING ---
PT_BINS = np.array([30, 40, 50, 60, 120, 200, 500])
ETA_BINS = np.array([-2.5, -1.5, -0.8, 0.0, 0.8, 1.5, 2.5])


# ==========================================
# 2. DATA EXTRACTION FUNCTIONS
# ==========================================
def getZhbbWeight(df_, year=None):
    if 'events_norm_weight' not in df_.columns:
        return pd.Series(1.0, index=df_.index) 

    weight = df_['events_norm_weight'].copy()
    gen_w = df_.get('events_genWeight', pd.Series(1.0, index=df_.index)).fillna(1.0)
    weight *= np.sign(gen_w)
    
    sfs = [
        'events_ele_reco_sf', 'events_ele_id_sf', 'events_mu_id_sf', 
        'events_mu_iso_sf', 'events_puWeight', 'events_topptWeight', 'events_mu_trig_sf'
    ]
    for sf in sfs:
        sf_col = df_.get(sf, pd.Series(1.0, index=df_.index)).fillna(1.0)
        weight *= sf_col
        
    return weight

def load_and_cut_data(variation='nominal', coffea_dir=COFFEA_DIR, year=2024):
    tracked_data = {proc: {} for proc in all_processes}
    all_files = glob.glob(os.path.join(coffea_dir, "*.coffea"))
    
    valid_files = [f for f in all_files if 'nom' in os.path.basename(f).lower()]
    print(f"Extracting '{variation}' from {len(valid_files)} matching files...")

    for file_path in valid_files:
        filein = load(file_path)
        genweight_dict = filein.get('sum_signOf_genweights', {})
        
        for raw_proc in filein['columns'].keys():
            mapped_proc = None
            # 1. Data handling
            if 'data' in raw_proc.lower() or 'DATA' in raw_proc:
                mapped_proc = 'data_obs'
            
            # 2. 5-Flavor Scheme (TTTo2L2Nu): Keep LF and C, Veto B
            elif 'TTTo2L2Nu' in raw_proc:
                if 'tt+B' in raw_proc: 
                    continue # Discard 5FS tt+B (redundant)
                mapped_proc = 'TTBar'
            
            # 3. 4-Flavor Scheme (TTbb_2L2Nu): Keep B, Veto LF and C
            elif 'TTbb_2L2Nu' in raw_proc:
                if 'tt+B' not in raw_proc: 
                    continue # Discard 4FS tt+LF/C (worse description)
                mapped_proc = 'tt_B'
            
            if mapped_proc not in all_processes: continue
                
            for dataset in filein['columns'][raw_proc].keys():
                genweight = genweight_dict.get(dataset, 1.0) 
                if isinstance(genweight, dict):
                    genweight = genweight.get(dataset, 1.0)
                
                try:
                    base_dict = filein['columns'][raw_proc][dataset]['btag_mask'][variation]
                except KeyError:
                    continue 
                    
                tmp_data = {}
                skip_dataset = False
                
                current_vars = validation_vars if mapped_proc == 'data_obs' else vars_to_extract
                
                for var in current_vars:
                    dict_key = f'spanet_output_{var}' if var in ['ttzbb', 'tthbb', 'ttbb', 'ttlf', 'ttcc', 'signal'] else var
                    
                    if dict_key in base_dict:
                        arr = np.array(base_dict[dict_key].value)
                    else:
                        continue
                        
                    if var == 'events_norm_weight' and mapped_proc != 'data_obs':
                        tmp_data[var] = arr / genweight
                    else:
                        tmp_data[var] = arr
                        
                if tmp_data: 
                    df = pd.DataFrame(tmp_data)
                    if dataset not in tracked_data[mapped_proc]:
                        tracked_data[mapped_proc][dataset] = []
                    tracked_data[mapped_proc][dataset].append(df)

    data_dict = {}
    for proc in all_processes:
        dfs_to_concat = []
        for dataset, branches in tracked_data[proc].items():
            dfs_to_concat.extend(branches)
            
        if dfs_to_concat:
            df = pd.concat(dfs_to_concat, ignore_index=True)
            
            # --- BASELINE HADRONIC CUTS ---
            #df = df[df['events_n_b_outZH'] >= 2] 
            #df = df[df['events_n_ak4jets'] >= 5]
            
            df['tot_weight'] = getZhbbWeight(df, year=year) if proc != 'data_obs' else 1.0
            data_dict[proc] = df
        else:
            data_dict[proc] = None 
            
    return data_dict

# ==========================================
# 3. PLOTTING FUNCTION
# ==========================================
def plot_annotated_heatmap(val_matrix, err_matrix, pt_bins, eta_bins, title, zlabel, filename, cmap='viridis'):
    """Plots a 2D grid with pT on the X-axis and Eta on the Y-axis."""
    fig, ax = plt.subplots(figsize=(14, 8))
    hep.style.use(hep.style.CMS)
    
    # Original shape: ny = pt_bins, nx = eta_bins
    ny, nx = val_matrix.shape
    
    # Create coordinate grid. X is pT, Y is Eta
    X, Y = np.meshgrid(pt_bins, eta_bins)
    
    # We transpose val_matrix so it matches the (eta, pt) shape of the grid
    im = ax.pcolormesh(X, Y, val_matrix.T, cmap=cmap, vmin=np.nanmin(val_matrix[val_matrix>0]), vmax=np.nanmax(val_matrix))
    cbar = fig.colorbar(im, ax=ax)
    cbar.set_label(zlabel, fontsize=18)
    
    # --- X-Axis (pT) ---
    ax.set_xscale('log') # Log scale moved to X-axis
    ax.set_xticks(pt_bins)
    ax.xaxis.set_major_formatter(ScalarFormatter())
    ax.set_xticklabels([f"{pt:.0f}" for pt in pt_bins])
    ax.set_xlabel(r"Electron $p_T$ [GeV]", fontsize=18)
    
    # --- Y-Axis (Eta) ---
    ax.set_yticks(eta_bins)
    ax.set_yticklabels([f"{eta:.1f}" for eta in eta_bins])
    ax.set_ylabel(r"Electron $\eta$", fontsize=18)
    
    ax.set_title(title, fontsize=20, pad=20)
    
    mid_val = (np.nanmax(val_matrix) + np.nanmin(val_matrix[val_matrix>0])) / 2.0
    
    # Loop over original matrix indices to annotate
    for i in range(ny): # i is pT index
        for j in range(nx): # j is Eta index
            val = val_matrix[i, j]
            err = err_matrix[i, j]
            
            if val > 0:
                text = f"{val:.3f}\n±{err:.3f}"
                if cmap in ['viridis']:
                    text_color = "black" if val > mid_val else "white"
                elif cmap == 'Blues':
                    text_color = "white" if val > mid_val else "black"
                else:
                    text_color = "black"
                    
                # X is pT (log scale -> geometric mean for visual center)
                x_center = np.exp((np.log(pt_bins[i]) + np.log(pt_bins[i+1])) / 2.0)
                # Y is Eta (linear scale -> arithmetic mean)
                y_center = (eta_bins[j] + eta_bins[j+1]) / 2.0
                    
                ax.text(x_center, y_center, text, ha='center', va='center', color=text_color, fontsize=11, fontweight='bold')
    
    hep.cms.label("Preliminary", data=True, lumi=110, ax=ax)
    plt.tight_layout()
    plt.savefig(filename)
    plt.close()
    print(f"Saved {filename}")

# ==========================================
# 4. SCALE FACTOR & SYSTEMATICS CALCULATION
# ==========================================
def calculate_trigger_sf(data_dict, target_mc='TTBar', target_data='data_obs'):
    print("\n--- Calculating Trigger Efficiencies & Systematics ---")
    results = {}
    
    for process in [target_data, target_mc]:
        df = data_dict[process]
        if df is None or df.empty:
            print(f"Skipping {process} - DataFrame is empty.")
            continue
            
        # RESTORED: Filter dataset based on reference triggers
        pass_ref = df[mu_triggers].any(axis=1)
        pass_tgt = df[ele_triggers].any(axis=1)
        pass_both = pass_ref & pass_tgt
        
        df_denom = df[pass_ref]
        df_num = df[pass_both]
        
        w_denom = df_denom['tot_weight'] if process != target_data else np.ones(len(df_denom))
        w_num = df_num['tot_weight'] if process != target_data else np.ones(len(df_num))
        
        hist_denom, _, _ = np.histogram2d(
            df_denom['events_ele_pt'], df_denom['events_ele_eta'], 
            bins=[PT_BINS, ETA_BINS], weights=w_denom
        )
        hist_denom_raw, _, _ = np.histogram2d(
            df_denom['events_ele_pt'], df_denom['events_ele_eta'], 
            bins=[PT_BINS, ETA_BINS]
        )
        
        hist_num, _, _ = np.histogram2d(
            df_num['events_ele_pt'], df_num['events_ele_eta'], 
            bins=[PT_BINS, ETA_BINS], weights=w_num
        )
        hist_num_raw, _, _ = np.histogram2d(
            df_num['events_ele_pt'], df_num['events_ele_eta'], 
            bins=[PT_BINS, ETA_BINS]
        )
        
        with np.errstate(divide='ignore', invalid='ignore'):
            efficiency = np.where(hist_denom > 0, hist_num / hist_denom, 0.0)
        
        lower_bound = stats.beta.ppf(0.3173 / 2, hist_num_raw, hist_denom_raw - hist_num_raw + 1)
        upper_bound = stats.beta.ppf(1 - 0.3173 / 2, hist_num_raw + 1, hist_denom_raw - hist_num_raw)
        
        stat_unc_up = np.nan_to_num(upper_bound, nan=1.0) - efficiency
        stat_unc_dn = efficiency - np.nan_to_num(lower_bound, nan=0.0)
        
        stat_err_sym = np.maximum(stat_unc_up, np.abs(stat_unc_dn))
        results[process] = {'eff': efficiency, 'stat_err': stat_err_sym}
        print(f"Calculated efficiency for {process}. Avg eff: {np.nanmean(efficiency):.3f}")

        if process == target_mc:
            w_base = df['tot_weight']
            
            hist_base, _, _ = np.histogram2d(df['events_ele_pt'], df['events_ele_eta'], bins=[PT_BINS, ETA_BINS], weights=w_base)
            # RESTORED: use pass_ref mask for hist_ref
            hist_ref, _, _  = np.histogram2d(df[pass_ref]['events_ele_pt'], df[pass_ref]['events_ele_eta'], bins=[PT_BINS, ETA_BINS], weights=df[pass_ref]['tot_weight'])
            hist_tgt, _, _  = np.histogram2d(df[pass_tgt]['events_ele_pt'], df[pass_tgt]['events_ele_eta'], bins=[PT_BINS, ETA_BINS], weights=df[pass_tgt]['tot_weight'])
            hist_both, _, _ = np.histogram2d(df[pass_both]['events_ele_pt'], df[pass_both]['events_ele_eta'], bins=[PT_BINS, ETA_BINS], weights=df[pass_both]['tot_weight'])
            
            with np.errstate(divide='ignore', invalid='ignore'):
                eff_ref = np.where(hist_base > 0, hist_ref / hist_base, 0.0)
                eff_tgt = np.where(hist_base > 0, hist_tgt / hist_base, 0.0)
                eff_both = np.where(hist_base > 0, hist_both / hist_base, 0.0)
                
                alpha = np.where(eff_both > 0, (eff_ref * eff_tgt) / eff_both, 1.0)
                results['alpha'] = alpha

    # ==========================================
    # STEP 6: Calculate Final Data/MC Ratio & Errors
    # ==========================================
    with np.errstate(divide='ignore', invalid='ignore'):
        sf_map = np.where(results[target_mc]['eff'] > 0, 
                          results[target_data]['eff'] / results[target_mc]['eff'], 0.0)
        
        rel_err_data = np.where(results[target_data]['eff'] > 0, results[target_data]['stat_err'] / results[target_data]['eff'], 0)
        rel_err_mc = np.where(results[target_mc]['eff'] > 0, results[target_mc]['stat_err'] / results[target_mc]['eff'], 0)
                              
        sf_stat_unc = sf_map * np.sqrt(rel_err_data**2 + rel_err_mc**2)
        
        # Apply Systematic Uncertainty
        syst_unc = np.abs(1.0 - results['alpha']) * sf_map
        
        # Total Uncertainty (Stat and Syst added in quadrature)
        sf_total_unc = np.sqrt(sf_stat_unc**2 + syst_unc**2)

    return results, sf_map, sf_stat_unc, syst_unc, sf_total_unc

import gzip
import correctionlib.schemav2 as schema

def export_to_correctionlib(sf_map, err_map, pt_bins, eta_bins, output_filename="electron_trigger_sf.json.gz"):
    print(f"\n--- Exporting to correctionlib ---")
    
    # 1. Define the input variables for the JSON
    input_pt = schema.Variable(name="pt", type="real", description="Electron pT")
    input_eta = schema.Variable(name="eta", type="real", description="Electron pseudorapidity")
    input_syst = schema.Variable(name="systematic", type="string", description="Systematic variation: 'nominal', 'up', 'down'")
    
    # 2. Calculate the 'up' and 'down' variations
    # We clip the 'down' variation at 0 to avoid negative weights
    sf_up = sf_map + err_map
    sf_dn = np.clip(sf_map - err_map, a_min=0.0, a_max=None)
    
    # 3. Helper to create a MultiBinning node
    # Note: numpy flattens row-major (C-style). Since our matrix is (pT, Eta), 
    # the inputs list must be ["pt", "eta"] to match the flattening order.
    def build_multibin(val_matrix):
        return schema.MultiBinning(
            nodetype="multibinning",
            inputs=["pt", "eta"],
            edges=[pt_bins.tolist(), eta_bins.tolist()],
            content=val_matrix.flatten().tolist(),
            # "clamp" means if an electron has pT > 500, it will automatically
            # receive the scale factor from the highest pT bin rather than crashing.
            flow="clamp" 
        )

    # 4. Create a Category node to handle the systematics mapping
    syst_category = schema.Category(
        nodetype="category",
        input="systematic",
        content=[
            schema.CategoryItem(key="nominal", value=build_multibin(sf_map)),
            schema.CategoryItem(key="up", value=build_multibin(sf_up)),
            schema.CategoryItem(key="down", value=build_multibin(sf_dn)),
        ]
    )

    # 5. Build the core Correction object
    correction = schema.Correction(
        name="EleTrigSF",
        description="Electron trigger efficiency scale factors (Data/MC)",
        version=1,
        inputs=[input_eta, input_pt, input_syst], 
        output=schema.Variable(name="weight", type="real", description="Scale factor weight"),
        data=syst_category
    )

    # 6. Wrap in a CorrectionSet and save to disk
    cset = schema.CorrectionSet(
        schema_version=2,
        description="Custom Trigger Scale Factors",
        corrections=[correction]
    )

    with gzip.open(output_filename, "wt") as fout:
        fout.write(cset.json(exclude_unset=True))
        
    print(f"Correction set successfully saved to: {output_filename}")


# ==========================================
# 5. EXECUTION
# ==========================================
if __name__ == "__main__":
    # 1. Load the data dictionary as usual
    data_dict = load_and_cut_data(variation='nominal')
    
    # 2. Combine the MC processes into one entry
    mc_to_combine = ['TTBar', 'tt_B']
    # Filter out None values in case one process failed to load
    valid_dfs = [data_dict[proc] for proc in mc_to_combine if data_dict.get(proc) is not None]

    if valid_dfs:
        # Create a new key 'MC_Combined' containing both 4FS and 5FS data
        data_dict['MC_Combined'] = pd.concat(valid_dfs, ignore_index=True)
        print(f"Successfully combined {mc_to_combine} into 'MC_Combined'.")
        print(f"Total MC events: {len(data_dict['MC_Combined'])}")
    else:
        print("Error: No MC DataFrames found to combine!")

    # 3. Calculate SF using 'MC_Combined' as your target MC
    results, sf_nominal, sf_stat_err, sf_syst_err, sf_total_err = calculate_trigger_sf(
        data_dict, 
        target_mc='MC_Combined',  # <-- Pointing to the new merged key
        target_data='data_obs'
    )
    
    # ==========================================
    # 6. OUTPUT & PLOTTING
    # ==========================================
    print("\n=== DERIVED NOMINAL SCALE FACTORS (pT vs Eta) ===")
    pt_labels = [f"pT {PT_BINS[i]}-{PT_BINS[i+1]}" for i in range(len(PT_BINS)-1)]
    eta_labels = [f"Eta {ETA_BINS[i]} to {ETA_BINS[i+1]}" for i in range(len(ETA_BINS)-1)]
    
    print("\nScale Factors (Data/MC):")
    print(pd.DataFrame(sf_nominal, index=pt_labels, columns=eta_labels).round(3))
    
    print("\nGenerating Heatmaps...")
    
    # Efficiency for Data
    plot_annotated_heatmap(
        results['data_obs']['eff'], results['data_obs']['stat_err'], 
        PT_BINS, ETA_BINS, 
        title="Data Trigger Efficiency", zlabel="Efficiency", 
        filename="Data_Efficiency_Heatmap.pdf", cmap='viridis'
    )
    
    # Efficiency for combined MC (5FS + 4FS)
    plot_annotated_heatmap(
        results['MC_Combined']['eff'], results['MC_Combined']['stat_err'], 
        PT_BINS, ETA_BINS, 
        title=r"Combined $t\bar{t}$ (4FS+5FS) MC Efficiency", zlabel="Efficiency", 
        filename="MC_Combined_Efficiency_Heatmap.pdf", cmap='viridis'
    )
    
    # Final Scale Factor
    plot_annotated_heatmap(
        sf_nominal, sf_total_err, 
        PT_BINS, ETA_BINS, 
        title="Electron Trigger Scale Factors (Combined MC)", zlabel="SF (Data / MC)", 
        filename="Trigger_ScaleFactor_Heatmap.pdf", cmap='viridis'
    )
    
    export_to_correctionlib(
        sf_map=sf_nominal, 
        err_map=sf_total_err, 
        pt_bins=PT_BINS, 
        eta_bins=ETA_BINS, 
        output_filename="ele_trig_sf_2024.json.gz"
    )

Extracting 'nominal' from 2 matching files...
Successfully combined ['TTBar', 'tt_B'] into 'MC_Combined'.
Total MC events: 475084

--- Calculating Trigger Efficiencies & Systematics ---
Calculated efficiency for data_obs. Avg eff: 0.869
Calculated efficiency for MC_Combined. Avg eff: 0.882

=== DERIVED NOMINAL SCALE FACTORS (pT vs Eta) ===

Scale Factors (Data/MC):
            Eta -2.5 to -1.5  Eta -1.5 to -0.8  Eta -0.8 to 0.0  \
pT 30-40               1.021             1.042            1.029   
pT 40-50               1.030             0.965            0.953   
pT 50-60               1.051             0.893            1.010   
pT 60-120              0.976             1.017            0.991   
pT 120-200             0.961             1.009            0.997   
pT 200-500             1.022             0.927            1.014   

            Eta 0.0 to 0.8  Eta 0.8 to 1.5  Eta 1.5 to 2.5  
pT 30-40             1.025           0.925           0.955  
pT 40-50             0.927           0.9

/tmp/ipykernel_1457301/3778176505.py:370: PydanticDeprecatedSince20: The `json` method is deprecated; use `model_dump_json` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  fout.write(cset.json(exclude_unset=True))
